# Vapor Pressure Curves

Computing and comparing vapor pressure curves for water using different IK-CAPE correlation models: Antoine, Kirchhoff, and Wagner equations.

## Setup

Import the correlation blocks and plotting libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathsim_chem.thermodynamics import Antoine, Kirchhoff, Wagner

## Antoine Equation

The Antoine equation is the most common three-parameter vapor pressure correlation:

$$\ln P_{\text{sat}} = a_0 - \frac{a_1}{T + a_2}$$

We use well-known coefficients for water (natural log, T in Kelvin, P in Pa).

In [ ]:
# Antoine coefficients for water (NIST, natural log form)
antoine = Antoine(a0=23.2256, a1=3835.18, a2=-45.343)

# Verify at the normal boiling point (373.15 K = 100 °C)
antoine.inputs[0] = 373.15
antoine.update(None)
P_sat = antoine.outputs[0]
print(f"P_sat at 100 °C: {P_sat:.0f} Pa (expected ~101325 Pa)")

## Kirchhoff Equation

The Kirchhoff equation is another three-parameter form:

$$\ln P_{\text{sat}} = a_0 - \frac{a_1}{T} - a_2 \ln(T)$$

It provides a slightly different functional form that can be more accurate over wider temperature ranges.

In [ ]:
# Kirchhoff coefficients for water (fitted to Antoine reference points)
kirchhoff = Kirchhoff(a0=50.4338, a1=6350.4068, a2=3.6963)

## Wagner Equation

The Wagner equation uses reduced temperature and is considered one of the most accurate vapor pressure correlations:

$$\ln P_{\text{sat}} = \ln P_c + \frac{1}{T_r}\left(a_2 \tau + a_3 \tau^{1.5} + a_4 \tau^{3} + a_5 \tau^{6}\right)$$

where $T_r = T / T_c$ and $\tau = 1 - T_r$. It requires the critical temperature and pressure as parameters.

In [ ]:
# Wagner coefficients for water
# Tc = 647.096 K, Pc = 22064000 Pa
wagner = Wagner(
    Tc=647.096, Pc=22064000,
    a2=-7.76451, a3=1.45838, a4=-2.77580, a5=-1.23303,
)

## Comparison

Evaluate all three correlations over a temperature range and plot the vapor pressure curves. Each block follows the standard PathSim interface: set the input, call `update`, read the output.

In [ ]:
# Temperature range: 300 K to 500 K
T_range = np.linspace(300, 500, 200)

def eval_over_range(block, T_range):
    """Evaluate a correlation block over an array of temperatures."""
    results = []
    for T in T_range:
        block.inputs[0] = T
        block.update(None)
        results.append(block.outputs[0])
    return np.array(results)

P_antoine = eval_over_range(antoine, T_range)
P_kirchhoff = eval_over_range(kirchhoff, T_range)
P_wagner = eval_over_range(wagner, T_range)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Linear scale
ax1.plot(T_range - 273.15, P_antoine / 1e3, label="Antoine")
ax1.plot(T_range - 273.15, P_kirchhoff / 1e3, "--", label="Kirchhoff")
ax1.plot(T_range - 273.15, P_wagner / 1e3, ":", label="Wagner")
ax1.axhline(101.325, color="gray", linestyle="-.", alpha=0.5, label="1 atm")
ax1.set_xlabel("Temperature [°C]")
ax1.set_ylabel("Vapor Pressure [kPa]")
ax1.set_title("Vapor Pressure of Water")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Clausius-Clapeyron plot (log P vs 1/T)
ax2.semilogy(1000 / T_range, P_antoine, label="Antoine")
ax2.semilogy(1000 / T_range, P_kirchhoff, "--", label="Kirchhoff")
ax2.semilogy(1000 / T_range, P_wagner, ":", label="Wagner")
ax2.set_xlabel("1000 / T [1/K]")
ax2.set_ylabel("Vapor Pressure [Pa]")
ax2.set_title("Clausius-Clapeyron Plot")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

All three correlations agree well in their common validity range. The Wagner equation is typically preferred for high-accuracy work since it is constrained to reach the critical point.